# Stage 05: Data Storage

**Project:** SPY Next-Day High-Volatility Risk Alert  
**Date:** 2026-08-23

This notebook completes the Stage 05 starter tasks with the real SPY OHLCV data acquired in Stage 04. It uses environment-driven paths, saves one typed DataFrame as raw CSV and processed Parquet, reloads both formats, validates the round trips, and demonstrates reusable suffix-routed storage utilities.

In [1]:
# Packages needed (run once in the Stage 02 environment if missing):
# %pip install pandas pyarrow python-dotenv

## 1. Environment-driven paths

The local `.env` defines `DATA_DIR_RAW=data/raw` and `DATA_DIR_PROCESSED=data/processed`. Relative settings are resolved against `homework05`, so behavior does not depend on where Jupyter was launched. The local `.env` is ignored; `.env.example` safely documents the required variable names.

In [2]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


def locate_homework_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == "homework05" and (candidate / "src").exists():
            return candidate
        nested = candidate / "homework" / "homework05"
        if (nested / "src").exists():
            return nested
    raise FileNotFoundError("Could not locate homework/homework05")


def resolve_env_path(homework_root: Path, variable: str, default: str) -> Path:
    configured = Path(os.getenv(variable, default)).expanduser()
    return configured if configured.is_absolute() else homework_root / configured


ROOT = locate_homework_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(dotenv_path=ROOT / ".env", override=True)
RAW = resolve_env_path(ROOT, "DATA_DIR_RAW", "data/raw")
PROCESSED = resolve_env_path(ROOT, "DATA_DIR_PROCESSED", "data/processed")
repo_gitignore = ROOT.parents[1] / ".gitignore"
env_is_ignored_by_rule = repo_gitignore.exists() and ".env" in repo_gitignore.read_text()

print("Homework root:", ROOT)
print("DATA_DIR_RAW ->", RAW)
print("DATA_DIR_PROCESSED ->", PROCESSED)
print("Local .env present:", (ROOT / ".env").exists())
print(".env covered by repository ignore rule:", env_is_ignored_by_rule)

Homework root: /Users/cengchengyu/Documents/NYU/Boot Camp/CS HW/Project/homework/homework05
DATA_DIR_RAW -> /Users/cengchengyu/Documents/NYU/Boot Camp/CS HW/Project/homework/homework05/data/raw
DATA_DIR_PROCESSED -> /Users/cengchengyu/Documents/NYU/Boot Camp/CS HW/Project/homework/homework05/data/processed
Local .env present: True
.env covered by repository ignore rule: True


In [3]:
from src.storage import (
    detect_format,
    get_parquet_engine,
    read_df,
    timestamp_utc,
    validate_roundtrip,
    write_df,
)

print("Available Parquet engine:", get_parquet_engine())

Available Parquet engine: pyarrow


## 2. Load the Stage 04 DataFrame

Rather than generate random sample data, this submission reuses the latest timestamped Stage 04 Nasdaq SPY snapshot. CSV cannot store dtype metadata, so the date and volume schema are supplied explicitly at read time. The code fails clearly if the prerequisite snapshot is absent.

In [4]:
stage04_raw = ROOT.parent / "homework04" / "data" / "raw"
source_candidates = sorted(stage04_raw.glob("api_NASDAQ_SPY_*.csv"))
if not source_candidates:
    raise FileNotFoundError(f"No Stage 04 SPY snapshot found in {stage04_raw}")
source_path = source_candidates[-1]

df = read_df(
    source_path,
    parse_dates=["date"],
    dtype={
        "open": "float64",
        "high": "float64",
        "low": "float64",
        "close": "float64",
        "volume": "int64",
    },
)
expected_columns = ["date", "open", "high", "low", "close", "volume"]
if list(df.columns) != expected_columns:
    raise ValueError(f"Unexpected Stage 04 schema: {list(df.columns)}")
if df.empty or df.isna().any().any() or not df["date"].is_unique:
    raise ValueError("Stage 04 input must be non-empty, complete, and date-unique")

print("Source snapshot:", source_path.relative_to(ROOT.parents[1]))
print("Shape:", df.shape)
display(df.head(3))
display(df.dtypes.rename("dtype").to_frame())

Source snapshot: homework/homework04/data/raw/api_NASDAQ_SPY_20260823-1503.csv
Shape: (2512, 6)


,date,open,high,low,close,volume
0,2016-08-23,219.25,219.60,218.90,218.97,53289030
1,2016-08-24,218.80,218.91,217.36,217.85,71553010
2,2016-08-25,217.40,218.19,217.22,217.70,69128600


,dtype
date,datetime64[us]
open,float64
high,float64
low,float64
close,float64
volume,int64


## 3. Save CSV and Parquet

`write_df` routes by suffix, creates missing parent directories, writes through a temporary file, and atomically replaces the destination. CSV is placed in the raw folder for portability; Parquet is placed in the processed folder for compact typed analytical access. At this stage, “processed” describes the storage representation—not additional data cleaning.

In [5]:
RUN_TIMESTAMP = timestamp_utc()
csv_path = RAW / f"spy_ohlcv_{RUN_TIMESTAMP}.csv"
parquet_path = PROCESSED / f"spy_ohlcv_{RUN_TIMESTAMP}.parquet"

write_df(df, csv_path)
write_df(df, parquet_path)

print("CSV format detected as:", detect_format(csv_path))
print("Parquet format detected as:", detect_format(parquet_path))
print("Saved CSV:", csv_path.relative_to(ROOT))
print("Saved Parquet:", parquet_path.relative_to(ROOT))

CSV format detected as: csv
Parquet format detected as: parquet
Saved CSV: data/raw/spy_ohlcv_20260823-1525.csv
Saved Parquet: data/processed/spy_ohlcv_20260823-1525.parquet


## 4. Reload and Validate

Both files are read through the same utility. Validation compares shape, column order, null counts, values, and critical dtype categories. The CSV reload receives schema hints because text files do not retain types; the Parquet reload restores its stored schema directly.

In [6]:
df_csv = read_df(
    csv_path,
    parse_dates=["date"],
    dtype={
        "open": "float64",
        "high": "float64",
        "low": "float64",
        "close": "float64",
        "volume": "int64",
    },
)
df_parquet = read_df(parquet_path)

critical_types = {
    "date": "datetime",
    "open": "float",
    "close": "float",
    "volume": "integer",
}
csv_validation = validate_roundtrip(df, df_csv, critical_types)
parquet_validation = validate_roundtrip(df, df_parquet, critical_types)

In [7]:
def validation_table(label: str, result: dict) -> pd.DataFrame:
    rows = [
        {"format": label, "check": name, "passed": passed, "detail": ""}
        for name, passed in result["checks"].items()
    ]
    rows.extend(
        {
            "format": label,
            "check": f"dtype:{column}",
            "passed": detail["passed"],
            "detail": f"expected {detail['expected_kind']}; got {detail['actual_dtype']}",
        }
        for column, detail in result["dtype_checks"].items()
    )
    return pd.DataFrame(rows)


validation_report = pd.concat(
    [
        validation_table("CSV", csv_validation),
        validation_table("Parquet", parquet_validation),
    ],
    ignore_index=True,
)
display(validation_report)
print("CSV validation passed:", csv_validation["passed"])
print("Parquet validation passed:", parquet_validation["passed"] )

,format,check,passed,detail
0,CSV,shape_equal,True,
1,CSV,column_order_equal,True,
2,CSV,null_counts_equal,True,
3,CSV,values_equal,True,
4,CSV,critical_columns_present,True,
5,CSV,critical_dtypes_valid,True,
6,CSV,dtype:date,True,expected datetime; got datetime64[us]
7,CSV,dtype:open,True,expected float; got float64
8,CSV,dtype:close,True,expected float; got float64
9,CSV,dtype:volume,True,expected integer; got int64


CSV validation passed: True
Parquet validation passed: True


In [8]:
size_comparison = pd.DataFrame(
    [
        {"format": "CSV", "bytes": csv_path.stat().st_size},
        {"format": "Parquet", "bytes": parquet_path.stat().st_size},
    ]
).assign(kib=lambda table: (table["bytes"] / 1024).round(2))
display(size_comparison)
print(
    "Parquet/CSV size ratio:",
    f"{parquet_path.stat().st_size / csv_path.stat().st_size:.3f}",
)

,format,bytes,kib
0,CSV,120977,118.14
1,Parquet,102744,100.34


Parquet/CSV size ratio: 0.849


## 5. Utility Error Behavior

Unsupported suffixes, nonexistent files, and missing Parquet engines receive actionable exceptions. The installed engine is reported above; the engine check in `get_parquet_engine` raises an installation command if neither PyArrow nor fastparquet is available.

In [9]:
for description, operation in [
    ("unsupported suffix", lambda: detect_format("example.xlsx")),
    ("missing CSV", lambda: read_df(RAW / "does_not_exist.csv")),
]:
    try:
        operation()
    except (ValueError, FileNotFoundError) as exc:
        print(f"Expected {description} error: {exc}")

Expected unsupported suffix error: Unsupported file suffix '.xlsx'; use .csv, .parquet, .parq, .pq
Expected missing CSV error: Data file does not exist: /Users/cengchengyu/Documents/NYU/Boot Camp/CS HW/Project/homework/homework05/data/raw/does_not_exist.csv


## 6. Storage Assumptions

- The latest filename-sorted Stage 04 SPY CSV is the intended prerequisite snapshot. Missing input stops the run; no synthetic fallback is generated.
- CSV prioritizes portability but requires explicit dtype restoration. Parquet prioritizes typed analytical access and compression but requires a compatible engine.
- Both outputs intentionally contain identical data. Stage 06, rather than this format-conversion exercise, will perform substantive preprocessing.
- Timestamp precision is one minute, matching the assignment example. A rerun within the same minute replaces the same logical output atomically.
- The local `.env` is present and covered by the repository ignore rule; only `.env.example` is committed.

In [10]:
assert csv_path.parent.resolve() == RAW.resolve()
assert parquet_path.parent.resolve() == PROCESSED.resolve()
assert csv_path.exists() and parquet_path.exists()
assert csv_validation["passed"] and parquet_validation["passed"]
assert validation_report["passed"].all()
assert env_is_ignored_by_rule

print("Stage 05 save/load, validation, utility, and path checks passed.")

Stage 05 save/load, validation, utility, and path checks passed.
